# Course project

This notebook includes the code for the course project in the DTU course [Integrated Energy Grids](https://kurser.dtu.dk/course/2024-2025/46770?menulanguage=en)  and is modelling and analyzing the german energy system. 
 
The report is provided along this notebook, further explaining the results. 

### Imports

In [13]:
import pandas as pd
import pypsa
import matplotlib.pyplot as plt
import numpy as np

In [14]:
import os
import sys

# Set path to the directory containing base.py
module_path = os.path.abspath(".")

if module_path not in sys.path:
    sys.path.append(module_path)

import base  # now you can use base.build_network()

from visualization import *

## E. Decarbonization
Select one target for decarbonization (i.e., one CO2 allowance limit). What is the CO2 price required to achieve that decarbonization level? Search for information on the existing CO2 tax in your country (if any) and discuss your results.

In [15]:
network = base.build_network(solve=False)

/Users/jonaswiendl/Documents/46770_IEG/46770_IEG/base.py:26: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  snapshots = pd.date_range(f"{year}-01-01", f"{year}-12-31 23:00", freq="H")
/Users/jonaswiendl/Documents/46770_IEG/46770_IEG/base.py:94: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_daily["Inflow [MW]"] = df_daily["Inflow [GWh]"] * 1000 / 24  # MW average per hour
/Users/jonaswiendl/Documents/46770_IEG/46770_IEG/base.py:96: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  inflow_ror_hourly = df_daily["Inflow [MW]"].resample("H").interpolate("linear")


Add CO2 constraint of german emissions in 2015

In [9]:
# check some numbers

print(f"System cost: {round(network.objective / 1e9, 2)} billion euros")
print("")
print("Optimal Generator Capactities in GW:")
print(network.generators.p_nom_opt.div(1e3))  # MW -> GW
print("")
print("Optimal Energy Generation in GWh/a")
print(network.generators_t.p.sum().div(1e6))

# Total emissions in the system (sum over all generators and time)
actual_emissions = (
    network.generators_t.p
    .multiply(network.generators.carrier.map(network.carriers["co2_emissions"]))
    .sum()
    .sum()
)

print("")
# The imposed CO₂ limit (from GlobalConstraint)
co2_limit = network.global_constraints.at["CO2Limit", "constant"]

print(f"Actual emissions: {actual_emissions:.2e} t")
print(f"CO₂ constraint  : {co2_limit:.2e} t")

dual = network.global_constraints.at["CO2Limit", "mu"]
print(f"Shadow price of CO₂: {dual:.2f} €/tCO2")

System cost: 37.6 billion euros

Optimal Generator Capactities in GW:
Generator
coal           -0.000000
lignite        -0.000000
biomass CHP     5.000000
OCGT           66.143381
ror             5.647002
onwind         20.827833
offwind        47.685581
solar          74.891179
Name: p_nom_opt, dtype: float64

Optimal Energy Generation in GWh/a
Generator
coal             0.000000
lignite          0.000000
biomass CHP     34.329163
OCGT           223.636364
ror             15.477029
onwind          21.343849
offwind        134.406264
solar           76.071895
dtype: float64

Actual emissions: 4.43e+07 t
CO₂ constraint  : 4.43e+07 t
Shadow price of CO₂: -365.22 €/tCO2


In [ ]:
# Introduce relative CO2 constraints 
co2_emissions_1990 = 369e6
emission_limits = [100, 1.0, 0.8, 0.6, 0.4, 0.2, 0.1, 0.0] 
emission_scenarios = [x*co2_emissions_1990 for x in emission_limits]

In [17]:
networks = []

for limit in emission_scenarios:
    n = base.build_network(solve=False)
    n.add("GlobalConstraint", "CO2Limit",
          type="primary_energy",
          carrier_attribute="co2_emissions",
          sense="<=",
          constant=limit)
    # Solve
    n.optimize(solver_name="gurobi")
    networks.append(n)

/Users/jonaswiendl/Documents/46770_IEG/46770_IEG/base.py:26: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  snapshots = pd.date_range(f"{year}-01-01", f"{year}-12-31 23:00", freq="H")
/Users/jonaswiendl/Documents/46770_IEG/46770_IEG/base.py:94: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_daily["Inflow [MW]"] = df_daily["Inflow [GWh]"] * 1000 / 24  # MW average per hour
/Users/jonaswiendl/Documents/46770_IEG/46770_IEG/base.py:96: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  inflow_ror_hourly = df_daily["Inflow [MW]"].resample("H").interpolate("linear")
Index(['DEU_elec'], dtype='object', name='Bus')
INFO:linopy.model: Solve problem usin

Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2625590


INFO:gurobipy:Set parameter LicenseID to value 2625590


...

INFO:gurobipy:


Solved in 23 iterations and 10.51 seconds (6.99 work units)


INFO:gurobipy:Solved in 23 iterations and 10.51 seconds (6.99 work units)


Optimal objective  1.543456132e+11


INFO:gurobipy:Optimal objective  1.543456132e+11
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 70088 primals, 148931 duals
Objective: 1.54e+11
Solver model: available
Solver message: 2

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-ext-p-lower, Generator-ext-p-upper were not assigned to the network.


In [ ]:
for network in networks: 
    dual = network.global_constraints.at["CO2Limit", "mu"]
    print(f"Shadow price of CO₂: {dual:.2f} €/tCO2")

In [31]:
co2_prices = []
for network in networks:
    co2_prices.append(network.global_constraints.at["CO2Limit", "mu"])

pd.DataFrame(index=emission_limits, data=co2_prices, columns=["Co2 price"])

,Co2 price
100.0,0.000000
1.0,0.000000
0.8,-13.300729
0.6,-14.455103
0.4,-15.201923
0.2,-15.966757
0.1,-765.475417
0.0,-167824.985584
